# BEC Raw Ingestion

Reads the BEC load group from `mde_dev.statedb.objectlist`, loops through each source CSV file, and creates Delta managed tables in `mde_dev.bec_raw`.

In [0]:
# Read the BEC load group from statedb.objectlist
bec_objects = spark.sql("""
    SELECT ObjectID, ObjectName, ExtractFileType, FileHeader,
           FilePath, FileColDelimiter, FileQuoteCharacter, FileEscapeCharacter,
           StageTableName
    FROM mde_dev.statedb.objectlist
    WHERE LoadGroup = 'BEC'
      AND IncludeInLoad = true
    ORDER BY ObjectID
""")

bec_rows = bec_objects.collect()
print(f"Found {len(bec_rows)} objects in BEC load group")
for row in bec_rows:
    print(f"  {row['ObjectName']:42s} -> {row['FilePath']}")

In [0]:
# Create the target schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS mde_dev.bec_raw")
print("Schema mde_dev.bec_raw ready")

In [0]:
# Loop through each object and create a Delta managed table
for row in bec_rows:
    table_name = row['StageTableName'] or row['ObjectName']
    file_path = row['FilePath']
    delimiter = row['FileColDelimiter'] or ','
    quote_char = row['FileQuoteCharacter'] or '"'
    escape_char = row['FileEscapeCharacter'] or '\\'
    has_header = row['FileHeader']
    
    full_table = f"mde_dev.bec_raw.{table_name}"
    
    # Read the CSV file using DataFrame API with full options from objectlist
    df = (spark.read
        .option("header", has_header)
        .option("delimiter", delimiter)
        .option("quote", quote_char)
        .option("escape", escape_char)
        .option("inferSchema", "true")
        .csv(file_path))
    
    # Drop existing table if present
    spark.sql(f"DROP TABLE IF EXISTS {full_table}")
    
    # Write as Delta managed table (default format for UC)
    df.write.mode("overwrite").saveAsTable(full_table)
    
    row_count = spark.table(full_table).count()
    print(f"\u2713 {full_table:50s} | rows: {row_count:>8,} | source: {file_path}")

print(f"\nAll {len(bec_rows)} tables created in mde_dev.bec_raw")